## Feautres to consider:

- Length
- Topology complexity (big difference between protons & kaons)
- Log likelihood (for dE/dx)

Ideas?
- 



The evidence pool:

- Have discriminating features, calculated directly from the original proton and kaon clusters
- Correlation matrix with latent dimensions
- Traverse each dimension and see what changes in the images
- Check linear predictions
- Check non-linear predictions
- Calculate features on reconstructed images and compare with original
- Show kaons sit in a different quadrant than protons on the calo, topo axes.
- Include muons.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import entropy, skew, kurtosis
from scipy.signal import peak_widths, find_peaks
from scipy.ndimage import uniform_filter1d
from skimage.measure import label, regionprops

In [ ]:
col = pd.read_pickle('/Volumes/easystore/proton-kaon/clusters/col.pkl')
ind = pd.read_pickle('/Volumes/easystore/proton-kaon/clusters/ind.pkl')

In [ ]:
col.columns

In [ ]:
def total_adc(image):
    return np.sum(image)

total_adc = [total_adc(a) for a in col.image_intensity]
col['total_adc'] = total_adc 

plt.hist(col[col['particle_type'] == 'proton']['total_adc'], bins=50)
plt.title("Total ADC Distribution for Protons")
plt.show()

plt.hist(col[col['particle_type'] == 'kaon']['total_adc'], bins=50)
plt.title("Total ADC Distribution for Kaons")
plt.show()


In [ ]:
def mean_adc(image):
    return np.mean(image)

mean_adc = [mean_adc(a) for a in col.image_intensity]
col['mean_adc'] = mean_adc 

plt.hist(col[col['particle_type'] == 'proton']['mean_adc'], bins=50)
plt.title("Mean ADC Distribution for Protons")
plt.show()

plt.hist(col[col['particle_type'] == 'kaon']['mean_adc'], bins=50)
plt.title("Mean ADC Distribution for Kaons")
plt.show()

In [ ]:
def median_adc(image):
    return np.median(image[image > 0])

median_adc = [median_adc(a) for a in col.image_intensity]
col['median_adc'] = median_adc 

plt.hist(col[col['particle_type'] == 'proton']['median_adc'], bins=50)
plt.title("Median ADC Distribution for Protons")
plt.show()

plt.hist(col[col['particle_type'] == 'kaon']['median_adc'], bins=50)
plt.title("Median ADC Distribution for Kaons")
plt.show()

In [ ]:
def max_adc(image):
    return np.max(image)

max_adc = [max_adc(a) for a in col.image_intensity]
col['max_adc'] = max_adc 

plt.hist(col[col['particle_type'] == 'proton']['max_adc'], bins=50)
plt.title("Max ADC Distribution for Protons")
plt.show()

plt.hist(col[col['particle_type'] == 'kaon']['max_adc'], bins=50)
plt.title("Max ADC Distribution for Kaons")
plt.show()

In [ ]:
def std_adc(image):
    return np.std(image[image > 0])

std_adc = [std_adc(a) for a in col.image_intensity]
col['std_adc'] = std_adc 

plt.hist(col[col['particle_type'] == 'proton']['std_adc'], bins=50)
plt.title("Standard Deviation of ADC Distribution for Protons")
plt.show()

plt.hist(col[col['particle_type'] == 'kaon']['std_adc'], bins=50)
plt.title("Standard Deviation of ADC Distribution for Kaons")
plt.show()

In [ ]:
def n_pixels(image):
    return np.sum(image > 0)

n_pixels = [n_pixels(a) for a in col.image_intensity]
col['n_pixels'] = n_pixels 

plt.hist(col[col['particle_type'] == 'proton']['n_pixels'], bins=50)
plt.title("Total Active Pixels for Protons")
plt.show()

plt.hist(col[col['particle_type'] == 'kaon']['n_pixels'], bins=50)
plt.title("Total Active Pixels for Kaons")
plt.show()

In [ ]:
def adc_entropy(image, n_bins=50):
    pixels = image[image > 0].ravel()
    if len(pixels) < 2:
        return np.nan
    counts, _ = np.histogram(pixels, bins=n_bins)
    n_occupied = (counts > 0).sum()
    if n_occupied < 2:
        return 0.0
    return entropy(counts) / np.log(n_occupied)

adc_entropy = [adc_entropy(a) for a in col.image_intensity]
col['adc_entropy'] = adc_entropy 

plt.hist(col[col['particle_type'] == 'proton']['adc_entropy'], bins=50)
plt.title("Shannon Entropy of ADC Distribution for Protons")
plt.show()

plt.hist(col[col['particle_type'] == 'kaon']['adc_entropy'], bins=50)
plt.title("Shannon Entropy of ADC Distribution for Kaons")
plt.show()

In [ ]:
def bragg_peak_height(column_maxes):
    return np.max(column_maxes)

bragg_peak_height = [bragg_peak_height(a) for a in col.column_maxes]
col['bragg_peak_height'] = bragg_peak_height 

plt.hist(col[col['particle_type'] == 'proton']['bragg_peak_height'], bins=50)
plt.title("Bragg Peak ADC for Protons")
plt.show()

plt.hist(col[col['particle_type'] == 'kaon']['bragg_peak_height'], bins=50)
plt.title("Bragg Peak ADC for Kaons")
plt.show()

In [ ]:
def bragg_peak_position(column_maxes):
    return np.argmax(column_maxes) / len(column_maxes)

bragg_peak_position = [bragg_peak_position(a) for a in col.column_maxes]
col['bragg_peak_position'] = bragg_peak_position 

plt.hist(col[col['particle_type'] == 'proton']['bragg_peak_position'], bins=50)
plt.title("Bragg Peak Position for Protons")
plt.show()

plt.hist(col[col['particle_type'] == 'kaon']['bragg_peak_position'], bins=50)
plt.title("Bragg Peak Position for Kaons")
plt.show()

In [ ]:
def bragg_peak_ratio(column_maxes):
    return np.max(column_maxes) / np.mean(column_maxes)

bragg_peak_ratio = [bragg_peak_ratio(a) for a in col.column_maxes]
col['bragg_peak_ratio'] = bragg_peak_ratio 

plt.hist(col[col['particle_type'] == 'proton']['bragg_peak_ratio'], bins=50)
plt.title("Bragg Peak Ratio Max:Mean for Protons")
plt.show()

plt.hist(col[col['particle_type'] == 'kaon']['bragg_peak_ratio'], bins=50)
plt.title("Bragg Peak Ratio Max:Mean for Kaons")
plt.show()

In [ ]:
def bragg_peak_to_median(column_maxes):
    return np.max(column_maxes) / np.median(column_maxes)

bragg_peak_to_median = [bragg_peak_to_median(a) for a in col.column_maxes]
col['bragg_peak_to_median'] = bragg_peak_to_median 

plt.hist(col[col['particle_type'] == 'proton']['bragg_peak_to_median'], bins=50)
plt.title("Bragg Peak Ratio Max:Median for Protons")
plt.show()

plt.hist(col[col['particle_type'] == 'kaon']['bragg_peak_to_median'], bins=50)
plt.title("Bragg Peak Ratio Max:Median for Kaons")
plt.show()

In [ ]:
def end_vs_start_ratio(column_maxes, p=0.1):
    n = len(column_maxes)
    k = int(p * n)
    return np.mean(column_maxes[-k:]) / np.mean(column_maxes[:k])

end_vs_start_ratio = [end_vs_start_ratio(a) for a in col.column_maxes]
col['end_vs_start_ratio'] = end_vs_start_ratio 

plt.hist(col[col['particle_type'] == 'proton']['end_vs_start_ratio'], bins=50)
plt.title("Bragg Peak Ratio Last:First for Protons")
plt.show()

plt.hist(col[col['particle_type'] == 'kaon']['bragg_peak_to_median'], bins=50)
plt.title("Bragg Peak Ratio Last:First for Kaons")
plt.show()

In [ ]:
def last_quartile_mean(column_maxes, p=0.25):
    n = len(column_maxes)
    k = int(p * n)
    return np.mean(column_maxes[-k:])

last_quartile_mean = [last_quartile_mean(a) for a in col.column_maxes]
col['last_quartile_mean'] = last_quartile_mean 

plt.hist(col[col['particle_type'] == 'proton']['last_quartile_mean'], bins=50)
plt.title("Last Quartile Mean for ADCs for Protons")
plt.show()

plt.hist(col[col['particle_type'] == 'kaon']['last_quartile_mean'], bins=50)
plt.title("Last Quartile Mean for ADCs for Kaons")
plt.show()

In [ ]:
def first_quartile_mean(column_maxes, p=0.25):
    n = len(column_maxes)
    k = int(p * n)
    return np.mean(column_maxes[:k])

first_quartile_mean = [first_quartile_mean(a) for a in col.column_maxes]
col['first_quartile_mean'] = first_quartile_mean 

plt.hist(col[col['particle_type'] == 'proton']['first_quartile_mean'], bins=50)
plt.title("First Quartile Mean for ADCs for Protons")
plt.show()

plt.hist(col[col['particle_type'] == 'kaon']['first_quartile_mean'], bins=50)
plt.title("First Quartile Mean for ADCs for Kaons")
plt.show()

In [ ]:
def bragg_rise_slope(column_maxes):
    x = np.arange((len(column_maxes)))
    slope, _ = np.polyfit(x, column_maxes, 1)
    return slope

bragg_rise_slope = [bragg_rise_slope(a) for a in col.column_maxes]
col['bragg_rise_slope'] = bragg_rise_slope 

plt.hist(col[col['particle_type'] == 'proton']['bragg_rise_slope'], bins=50)
plt.title("Slope of ADC rise for Protons")
plt.show()

plt.hist(col[col['particle_type'] == 'kaon']['bragg_rise_slope'], bins=50)
plt.title("Slope of ADC rise for Kaons")
plt.show()

In [ ]:
def peak_integral_fraction(column_maxes, p=0.15):
    n = len(column_maxes)
    k = int(p * n)
    end = np.sum(column_maxes[column_maxes > 0][-k:])
    total = np.sum(column_maxes[column_maxes > 0])
    return end / total

peak_integral_fraction = [peak_integral_fraction(a) for a in col.column_maxes]
col['peak_integral_fraction'] = peak_integral_fraction 

plt.hist(col[col['particle_type'] == 'proton']['peak_integral_fraction'], bins=50)
plt.title("Last 15 over total ADC for Protons")
plt.show()

plt.hist(col[col['particle_type'] == 'kaon']['peak_integral_fraction'], bins=50)
plt.title("Last 15 over total ADC for Kaons")
plt.show()

In [ ]:
def bragg_peak_width(column_maxes):
    peak_idx = np.argmax(column_maxes)
    widths, _, _, _ = peak_widths(column_maxes, [peak_idx], rel_height=0.5)
    return widths[0]

bragg_peak_width = [bragg_peak_width(a) for a in col.column_maxes]
col['bragg_peak_width'] = bragg_peak_width 

plt.hist(col[col['particle_type'] == 'proton']['bragg_peak_width'], bins=50)
plt.title("Bragg width for Protons")
plt.show()

plt.hist(col[col['particle_type'] == 'kaon']['bragg_peak_width'], bins=50)
plt.title("Bragg width for Kaons")
plt.show()

In [ ]:
def profile_skewness(column_maxes):
    return skew(column_maxes[column_maxes > 0])

profile_skewness = [profile_skewness(a) for a in col.column_maxes]
col['profile_skewness'] = profile_skewness 

plt.hist(col[col['particle_type'] == 'proton']['profile_skewness'], bins=50)
plt.title("Skewness for Protons")
plt.show()

plt.hist(col[col['particle_type'] == 'kaon']['profile_skewness'], bins=50)
plt.title("Skewness for Kaons")
plt.show()

In [ ]:
def profile_kurtosis(column_maxes):
    return kurtosis(column_maxes[column_maxes > 0])

profile_kurtosis = [profile_kurtosis(a) for a in col.column_maxes]
col['profile_kurtosis'] = profile_kurtosis 

plt.hist(col[col['particle_type'] == 'proton']['profile_kurtosis'], bins=50)
plt.title("Kurtosis for Protons")
plt.show()

plt.hist(col[col['particle_type'] == 'kaon']['profile_kurtosis'], bins=50)
plt.title("Kurtosis for Kaons")
plt.show()

In [ ]:
def profile_cv(column_maxes):
    return np.std(column_maxes[column_maxes > 0]) / np.mean(column_maxes[column_maxes > 0])

profile_cv = [profile_cv(a) for a in col.column_maxes]
col['profile_cv'] = profile_cv 

plt.hist(col[col['particle_type'] == 'proton']['profile_cv'], bins=50)
plt.title("Coefficient of variation for Protons")
plt.show()

plt.hist(col[col['particle_type'] == 'kaon']['profile_cv'], bins=50)
plt.title("Coefficient of variation for Kaons")
plt.show()

In [ ]:
def n_local_maxima(column_maxes):
    peaks, _ = find_peaks(column_maxes)
    return len(peaks)

n_local_maxima = [n_local_maxima(a) for a in col.column_maxes]
col['n_local_maxima'] = n_local_maxima 

plt.hist(col[col['particle_type'] == 'proton']['n_local_maxima'], bins=50)
plt.title("No. of Local Maxima for Protons")
plt.show()

plt.hist(col[col['particle_type'] == 'kaon']['n_local_maxima'], bins=50)
plt.title("No. of Local Maxima for Kaons")
plt.show()

In [ ]:
def monotonic_rise_fraction(column_maxes, smooth=3):
    cm = uniform_filter1d(column_maxes.astype(float), size=smooth)
    if np.argmax(cm) < len(cm) / 2:
        cm = cm[::-1]
    diffs = np.diff(cm)
    return (diffs > 0).sum() / len(diffs)

monotonic_rise_fraction = [monotonic_rise_fraction(a) for a in col.column_maxes]
col['monotonic_rise_fraction'] = monotonic_rise_fraction 

plt.hist(col[col['particle_type'] == 'proton']['monotonic_rise_fraction'], bins=50)
plt.title("Fraction of consecutive wire pairs where dE/dx increases for Protons")
plt.show()

plt.hist(col[col['particle_type'] == 'kaon']['monotonic_rise_fraction'], bins=50)
plt.title("Fraction of consecutive wire pairs where dE/dx increases for Kaons")
plt.show()

In [ ]:
def relative_peak_energy(column_maxes):
    i = np.argmax(column_maxes)
    l = int(len(column_maxes) * 0.1)
    window = column_maxes[max(0, i-l): i+l+1]
    peak = np.sum(window)
    total = np.sum(column_maxes[column_maxes>0])
    return peak / total

relative_peak_energy = [relative_peak_energy(a) for a in col.column_maxes]
col['relative_peak_energy'] = relative_peak_energy 

plt.hist(col[col['particle_type'] == 'proton']['relative_peak_energy'], bins=50)
plt.title("Relative Peak Energy for Protons")
plt.show()

plt.hist(col[col['particle_type'] == 'kaon']['relative_peak_energy'], bins=50)
plt.title("Relative Peak Energy for Kaons")
plt.show()

In [ ]:
def lag1_autocorr(x):
    x = np.asarray(x)
    if len(x) < 2:
        return np.nan
    x1 = x[:-1]
    x2 = x[1:]
    if np.std(x1) == 0 or np.std(x2) == 0:
        return np.nan
    return np.corrcoef(x1, x2)[0, 1]

lag1_autocorr = [lag1_autocorr(a) for a in col.column_maxes]
col['lag1_autocorr'] = lag1_autocorr 

plt.hist(col[col['particle_type'] == 'proton']['lag1_autocorr'], bins=50)
plt.title("Lag-1 autocorrelation for Protons")
plt.show()

plt.hist(col[col['particle_type'] == 'kaon']['lag1_autocorr'], bins=50)
plt.title("Lag-1 autocorrelation for Kaons")
plt.show()

In [ ]:
def solidity(image_intensity, threshold=0):
    binary = image_intensity > threshold
    labeled = label(binary)
    if labeled.max() == 0:
        return np.nan
    regions = regionprops(labeled)
    main = max(regions, key=lambda r: r.area)  # largest connected region
    return main.solidity

solidity = [solidity(a) for a in col.image_intensity]
col['solidity'] = solidity 

plt.hist(col[col['particle_type'] == 'proton']['solidity'], bins=50)
plt.title("Solidity for Protons")
plt.show()

plt.hist(col[col['particle_type'] == 'kaon']['solidity'], bins=50)
plt.title("Solidity for Kaons")
plt.show()